In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_from_disk

# Update this if your Drive folder name changes.
DATASET_ROOT = "/content/drive/MyDrive/Math Olympiad Competition/Experimentation/processed_splits/omr_aimo3"
PREFERRED_SPLIT = "100"

print("Dataset root:", DATASET_ROOT)
print("Preferred split:", PREFERRED_SPLIT)

Dataset root: /content/drive/MyDrive/Math Olympiad Competition/Experimentation/processed_splits/omr_aimo3
Preferred split: 100


In [9]:
# Transform chosen prepared split into a DataFrame for easier analysis.

def resolve_prepared_dataset_path(root: str, preferred_split: str = "100") -> str:
    tokenized_candidate = os.path.join(root, "splits_tokenized", preferred_split)
    raw_candidate = os.path.join(root, "splits", preferred_split)

    if os.path.exists(tokenized_candidate):
        print("Using tokenized dataset:", tokenized_candidate)
        return tokenized_candidate
    if os.path.exists(raw_candidate):
        print("Using raw dataset:", raw_candidate)
        return raw_candidate

    raise FileNotFoundError(
        "Could not find prepared dataset. Checked:\n"
        f"- {tokenized_candidate}\n"
        f"- {raw_candidate}"
    )


def dataset_to_df(ds_split):
    df_local = ds_split.to_pandas()

    # Normalize common field names for easier downstream EDA.
    rename_map = {}
    if "problem" in df_local.columns:
        rename_map["problem"] = "question"
    if "expected_answer" in df_local.columns:
        rename_map["expected_answer"] = "answer"
    if "generated_solution" in df_local.columns:
        rename_map["generated_solution"] = "solution"

    return df_local.rename(columns=rename_map)


print("Loading dataset split...")
dataset_path = resolve_prepared_dataset_path(DATASET_ROOT, PREFERRED_SPLIT)
prepared = load_from_disk(dataset_path)

ds_split = prepared["train"] if hasattr(prepared, "keys") and "train" in prepared else prepared

df = dataset_to_df(ds_split)
print("Dataset loaded. Number of samples:", len(df))
print("Columns:", list(df.columns))

Loading dataset split...
Using tokenized dataset: /content/drive/MyDrive/Math Olympiad Competition/Experimentation/processed_splits/omr_aimo3/splits_tokenized/100
Dataset loaded. Number of samples: 70
Columns: ['answer', 'input_ids', 'attention_mask', 'labels']


In [13]:
print("\nSample data:")
display(df)


Sample data:


,answer,input_ids,attention_mask,labels
0,50,"[2610, 525, 264, 35972, 3491, 28961, 624, 50, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-100
1,291,"[2610, 525, 264, 35972, 3491, 28961, 624, 50, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-100
2,40,"[2610, 525, 264, 35972, 3491, 28961, 624, 50, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-100
3,1,"[2610, 525, 264, 35972, 3491, 28961, 624, 50, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-100
4,72,"[2610, 525, 264, 35972, 3491, 28961, 624, 50, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-100
...,...,...,...,...
65,15120,"[2610, 525, 264, 35972, 3491, 28961, 624, 50, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-100
66,105,"[2610, 525, 264, 35972, 3491, 28961, 624, 50, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-100
67,3,"[2610, 525, 264, 35972, 3491, 28961, 624, 50, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-100
68,105,"[2610, 525, 264, 35972, 3491, 28961, 624, 50, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-100
